# Общий notebook ddG-экспериментов

Notebook собирает текущий FireProt-like датасет, внешний S669 benchmark и быстрый сравнительный анализ опубликованных S669 predictions против experimental ddG. Тяжелые локальные пересчеты FoldX/Rosetta/OpenFold остаются отдельным этапом, потому что требуют бинарей, GPU/runtime и занимают существенно больше времени.

In [ ]:
# Cell 0 - CONFIG
from pathlib import Path
import logging
import sys

import pandas as pd

repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent

sys.path.insert(0, str(repo_root))

dataset_path = repo_root / 'data' / 'processed' / 'dataset_v1.csv'
s669_path = repo_root / 'data' / 'processed' / 's669_dataset_v1.csv'
analysis_dir = repo_root / 'analysis' / 's669_benchmark_analysis'

logging.basicConfig(level=logging.INFO, format='%(levelname)s:%(name)s:%(message)s')
print(repo_root)


## 1. Основной FireProt-like датасет

Этот блок оставлен как основной внутренний набор эксперимента. Он нужен для текущей статьи как Dataset A.

In [ ]:
# Cell 1 - FIREPROT-LIKE DATASET
from pipeline.dataset import DatasetConfig, check_dataset_exists, download_dataset, preprocess_dataset

fireprot_config = DatasetConfig(processed_path=dataset_path)
if not check_dataset_exists(fireprot_config):
    download_dataset(fireprot_config)
    preprocess_dataset(fireprot_config)

fireprot_df = pd.read_csv(dataset_path)
print(f'FireProt-like rows: {len(fireprot_df)}')
print(f'FireProt-like proteins: {fireprot_df["protein_id"].nunique()}')
display(fireprot_df['experimental_ddg'].describe())
display(fireprot_df.head())


## 2. Внешний benchmark S669

S669 используется как независимая проверка обобщаемости. Direct variants импортируются из WT structures, inverse variants - из предоставленных inverse/mutant structures. Строки с несовпадением WT residue в фактической структуре отбрасываются и сохраняются в audit CSV.

In [ ]:
# Cell 2 - S669 IMPORT
from pipeline.external_benchmarks import S669Config, check_s669_exists, preprocess_s669, load_s669

s669_config = S669Config()
if not check_s669_exists(s669_config):
    preprocess_s669(s669_config)

s669_df = load_s669(s669_config)
print(f'S669 imported rows: {len(s669_df)}')
print(f'S669 WT PDBs: {s669_df["pdb_id"].nunique()}')
display(s669_df.groupby('direction').size().rename('rows'))
display(s669_df['experimental_ddg'].describe())
display(s669_df.head())


## 3. Сравнение готовых S669 predictions с experimental ddG

Исходный S669 CSV уже содержит predictions ряда опубликованных методов. Этот блок приводит их к единому long-format и считает метрики против experimental ddG: Pearson, Spearman, RMSE, MAE, bias и sign accuracy.

In [ ]:
# Cell 3 - S669 PUBLISHED METHOD COMPARISON
from pipeline.s669_analysis import analyze_s669_published_predictions

s669_outputs = analyze_s669_published_predictions()
metrics = pd.read_csv(s669_outputs['metrics'])
coverage = pd.read_csv(s669_outputs['coverage'])

print('S669 analysis outputs:')
for name, path in s669_outputs.items():
    print(f'  {name}: {path}')

display(metrics.head(20))
display(coverage.head(40))


## 4. Быстрый вывод по S669

Этот блок фиксирует машинно-считаемый summary, который удобно переносить в план статьи. Experimental DDG остается эталоном; опубликованные method columns рассматриваются как predictions.

In [ ]:
# Cell 4 - S669 SUMMARY
top = metrics.sort_values(['spearman', 'pearson', 'sign_accuracy'], ascending=False).head(8)
print('Top S669 methods by Spearman:')
display(top[['method', 'representation', 'n', 'pearson', 'spearman', 'mae', 'sign_accuracy']])

foldx_row = metrics.loc[metrics['method'].eq('FoldX')]
if not foldx_row.empty:
    display(foldx_row[['method', 'representation', 'n', 'pearson', 'spearman', 'mae', 'sign_accuracy']])

print('Interpretation: S669 is an external benchmark for method robustness; it does not directly answer OpenFold-vs-experimental structure sensitivity until local OpenFold/FoldX/Rosetta runs are added for these cases.')


## 5. Optional: локальный пересчет S669 через project ddG harness

Полный пересчет всех S669 cases локальными методами может быть тяжелым. Для smoke/protocol check начните с малого `sample_n`, затем масштабируйте через отдельный batch runner или scheduler.

In [ ]:
# Cell 5 - LOCAL HARNESS SMOKE TEMPLATE
# This cell intentionally does not run the full S669 benchmark automatically.
# It prepares command lines for a small local smoke sample using the existing harness.

sample_n = 3
sample = s669_df.head(sample_n).copy()
commands = []
for _, row in sample.iterrows():
    case_id = f"S669_{row['protein_id']}_{row['mutation_id']}"
    mutation = f"{row['chain']}:{row['wt_residue']}{row['pdb_residue_id']}{row['mut_residue']}"
    output = repo_root / 'analysis' / 's669_local_harness_smoke' / f"{case_id}.json"
    commands.append(
        ' '.join([
            'PYTHONPATH=openfold-3',
            'python3 openfold-3/scripts/dev/run_ddg_benchmark_harness.py',
            f'--case-id {case_id}',
            f'--structure "{row["pdb_path"]}"',
            f'--mutation {mutation}',
            f'--experimental-ddg {row["experimental_ddg"]}',
            f'--output "{output}"',
        ])
    )

for command in commands:
    print(command)
